In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    "http://localhost:9200",
)

print(es.info())

{'name': 'ahmed-HP-EliteBook-840-G1', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'W4B7ErSDS-G_v4p6lxl0sg', 'version': {'number': '9.3.2', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '43a703737aab6baefa748bc7b69e4054926f2b2c', 'build_date': '2026-03-16T13:12:56.143057855Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [2]:
index_name = "sales"

if not es.indices.exists(index=index_name):
    es.indices.create(index=index_name)

print("Index créé")

Index créé


In [7]:
data = [
    {"product": "Laptop", "price": 1200, "quantity": 3, "category": "IT"},
    {"product": "Phone", "price": 800, "quantity": 5, "category": "IT"},
    {"product": "Chair", "price": 150, "quantity": 10, "category": "Furniture"},
    {"product": "Desk", "price": 300, "quantity": 3, "category": "IT"},
]

for i, doc in enumerate(data):
    es.index(index="sales", id=i, document=doc)
print("Données insérées")

Données insérées


In [8]:
res = es.search(index="sales", query={
    "match": {"category": "IT"}
})

for hit in res["hits"]["hits"]:
    print(hit["_source"])

{'product': 'Desk', 'price': 300, 'quantity': 3, 'category': 'IT'}
{'product': 'Laptop', 'price': 1200, 'quantity': 3, 'category': 'IT'}
{'product': 'Phone', 'price': 800, 'quantity': 5, 'category': 'IT'}


In [11]:
res = es.search(
    index="sales",
    size=0,
    aggs={
        "total_by_category": {
            "terms": {"field": "category.keyword"},
            "aggs": {
                "total_price": {"sum": {"field": "price"}}
            }
        }
    }
)

print(res["aggregations"])

{'total_by_category': {'doc_count_error_upper_bound': 0, 'sum_other_doc_count': 0, 'buckets': [{'key': 'IT', 'doc_count': 3, 'total_price': {'value': 2300.0}}, {'key': 'Furniture', 'doc_count': 1, 'total_price': {'value': 150.0}}]}}


In [14]:
import pandas as pd 

df = pd.DataFrame(data)
print(df)

print(df.groupby("category")["price"].sum())

  product  price  quantity   category
0  Laptop   1200         3         IT
1   Phone    800         5         IT
2   Chair    150        10  Furniture
3    Desk    300         3         IT
category
Furniture     150
IT           2300
Name: price, dtype: int64
